In [ ]:
# -*- coding: utf-8 -*-


BhuvanFitterPierce.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1avypFL8nCLyVYymOm8yad9WnD9OvlqQV

# BhuvanFitter — Distribution Fitting & Truncation Index for Gene Expression Data

This notebook implements **BhuvanFitter**, a tool for fitting parametric distributions to
gene expression histograms and computing a **Truncation Index (TI)** that quantifies how
strongly overexpression is constrained by lethality.

---

## Biological Motivation

In a pooled CRISPR / transposon overexpression screen, each strain carries a different
insertion that drives expression of a target gene at a different level. When high expression
is lethal, strains with very high expression drop out of the population — their expression
values are never observed. This creates a **right-truncated distribution**: the right tail
of the true expression distribution is cut off at some ceiling `x_max`.

The **Truncation Index** measures how close that ceiling is to the peak of the distribution.
A gene whose ceiling sits very close to its peak is under strong lethality constraint
(overexpression is toxic). A gene whose ceiling is far out in the tail is essentially
unconstrained.

---

## Two Fitting Approaches

| Approach | Method | Pros | Cons |
|---|---|---|---|
| `fourparam` | 4-parameter Gaussian fit to histogram counts (NLS) | Fast, visually intuitive | Biased — fitted to truncated data |
| `mle` | Right-truncated Gaussian via Maximum Likelihood | Recovers the **true** untruncated μ and σ | Slower, requires optimization |

---

## Truncation Index Metrics

Each fitting approach produces **two complementary truncation index metrics**.
The naming convention is parallel across both models:

| Metric | fourparam column | mle column | Formula | Scale |
|---|---|---|---|---|
| **σ-distance** | `ti_fourparam_sigma_dist` | `ti_mle_sigma_dist` | `(x_max − μ) / σ` | Unbounded, σ units |
| **Height ratio** | `ti_fourparam_height_ratio` | `ti_mle_height_ratio` | `f(x_max) / f(peak)` | [0, 1] |

Each batch function runs **only its own model** and outputs only its own metrics.
The two tables are independent and can be joined on `gene` if needed.

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, minimize
from scipy.stats import norm
from scipy.special import ndtr          # fast, numerically stable normal CDF
import pandas as pd
import warnings


## 2. 4-Parameter Gaussian Model

The histogram-based fit uses a **4-parameter Gaussian**:

$$y = y_0 + A \cdot \exp\!\left(-\left(\frac{x - x_0}{w}\right)^2\right)$$

| Parameter | Meaning |
|---|---|
| `y0` | Baseline offset (accounts for background counts) |
| `A` | Amplitude — peak height above baseline |
| `x0` | Centre of the peak |
| `w` | Width parameter; related to σ by `w = σ√2` |

This function is defined at **module level** (not inside a class) because `scipy.optimize.curve_fit`
requires a plain callable — it cannot pickle instance methods.

In [ ]:
def _fourparam_gaussian(x, y0, A, x0, w):
    """
    4-parameter Gaussian model used by curve_fit.

        y = y0 + A * exp(-((x - x0) / w)^2)

    Parameters
    ----------
    x  : array-like  Input x values.
    y0 : float       Baseline offset.
    A  : float       Amplitude (peak height above baseline).
    x0 : float       Centre of the peak.
    w  : float       Width parameter (w = sigma * sqrt(2)).
    """
    return y0 + A * np.exp(-((x - x0) / w) ** 2)


## 3. MLE Helpers for the Right-Truncated Gaussian

### Why truncated MLE?

If we simply fit a Gaussian to the observed (truncated) data, both the mean and
standard deviation will be **underestimated** — the missing right tail pulls the
estimates toward the centre. To recover the *true* underlying parameters we must
explicitly model the truncation.

### The truncated Gaussian likelihood

For data drawn from $N(\mu, \sigma^2)$ subject to $x \le x_{\max}$, the
conditional density is:

$$f(x \mid x \le x_{\max}) = \frac{\phi\!\left(\frac{x-\mu}{\sigma}\right)}{\sigma \cdot \Phi\!\left(\frac{x_{\max}-\mu}{\sigma}\right)}$$

where $\phi$ is the standard normal PDF and $\Phi$ is the standard normal CDF.
Maximising the sum of log-densities over all observed data points gives MLE
estimates of the **true** $\mu$ and $\sigma$.

In [ ]:
def _truncated_gaussian_nll(params, data, x_max):
    """
    Negative log-likelihood of a right-truncated Gaussian.

    The observed data are draws from N(mu, sigma) conditioned on x <= x_max.
    Any strain with expression > x_max is unobserved (lethal / dropped out).

    Log-likelihood per observation:
        log f(x | x <= x_max) = log phi((x - mu)/sigma)
                                 - log sigma
                                 - log Phi((x_max - mu)/sigma)

    Parameters
    ----------
    params : (mu, log_sigma)
        log_sigma is optimised instead of sigma directly to enforce sigma > 0
        without box constraints.
    data   : np.ndarray   Observed expression values (all <= x_max).
    x_max  : float        Truncation point (observed maximum).

    Returns
    -------
    float   Negative log-likelihood (scalar to be minimised).
    """
    mu, log_sigma = params
    sigma = np.exp(log_sigma)          # always positive

    z     = (data - mu) / sigma
    z_max = (x_max - mu) / sigma

    log_phi = -0.5 * z ** 2 - 0.5 * np.log(2 * np.pi) - log_sigma
    log_Phi = np.log(ndtr(z_max) + 1e-300)   # 1e-300 guards against log(0)

    nll = -np.sum(log_phi - log_Phi)
    return nll


def _fit_mle_truncated(data, x_max):
    """
    Fit a right-truncated Gaussian via MLE.

    Uses Nelder-Mead as the primary optimiser (robust, derivative-free)
    and falls back to BFGS if Nelder-Mead reports failure.

    Parameters
    ----------
    data  : np.ndarray   Observed expression values.
    x_max : float        Truncation point.

    Returns
    -------
    mu    : float   MLE estimate of the true (untruncated) mean.
    sigma : float   MLE estimate of the true (untruncated) std.
    nll   : float   Negative log-likelihood at the optimum.

    Raises
    ------
    RuntimeError if both optimisers fail.
    """
    mu0    = float(np.mean(data))
    sigma0 = float(np.std(data, ddof=1))
    if sigma0 < 1e-8:
        sigma0 = 1e-8
    p0 = [mu0, np.log(sigma0)]

    result = minimize(
        _truncated_gaussian_nll,
        p0,
        args=(data, x_max),
        method="Nelder-Mead",
        options={"xatol": 1e-8, "fatol": 1e-8, "maxiter": 50_000},
    )

    if not result.success:
        result = minimize(
            _truncated_gaussian_nll,
            p0,
            args=(data, x_max),
            method="BFGS",
            options={"gtol": 1e-6, "maxiter": 10_000},
        )

    if not result.success:
        raise RuntimeError(
            f"MLE truncated Gaussian optimisation failed: {result.message}"
        )

    mu_mle    = float(result.x[0])
    sigma_mle = float(np.exp(result.x[1]))
    nll       = float(result.fun)
    return mu_mle, sigma_mle, nll


## 4. The `BhuvanFitter` Class

`BhuvanFitter` is the main interface. It:

1. Cleans and stores the raw expression data for one gene.
2. Pre-computes and caches the histogram (so it is not recomputed on every plot).
3. Exposes `.fit("fourparam")` and `.fit("mle")` to run either fitting approach.
4. Exposes truncation index properties with a **consistent naming convention** across both models.
5. Provides a `.hist()` method for visualisation with optional curve overlays.

### Truncation index property naming

Both models expose the same two TI flavours under parallel names:

| Flavour | fourparam property | mle property |
|---|---|---|
| σ-distance | `ti_fourparam_sigma_dist` | `ti_mle_sigma_dist` |
| Height ratio | `ti_fourparam_height_ratio` | `ti_mle_height_ratio` |

**σ-distance**: how many standard deviations of the fitted distribution `x_max` lies above the fitted peak. Lower = more truncated.

**Height ratio**: the fitted curve's value at `x_max` divided by its value at the peak. Equivalent to `exp(−sigma_dist²)` for a Gaussian. Bounded [0, 1]. Higher = more truncated (ceiling is near the peak).

In [ ]:
class BhuvanFitter:
    """
    Fits parametric distributions to gene expression histograms and computes
    Truncation Indices that quantify overexpression lethality constraints.

    Two fitting approaches
    ----------------------
    ``"fourparam"``
        4-parameter Gaussian fitted to histogram bin counts via nonlinear
        least squares (TRF method, robust soft-L1 loss). Fast visual fit,
        but parameters are biased because the fit is anchored to already-
        truncated data.

    ``"mle"``
        Right-truncated Gaussian fitted via Maximum Likelihood Estimation
        on the raw data. Recovers the TRUE underlying mu and sigma by
        explicitly accounting for the missing right tail above x_max.

    Truncation index properties
    ---------------------------
    Both models expose two metrics under parallel names:

        ti_fourparam_sigma_dist    (x_max - x0) / (w / sqrt(2))   [fourparam]
        ti_fourparam_height_ratio  f(x_max) / f(x0)               [fourparam]

        ti_mle_sigma_dist          (x_max - mu_mle) / sigma_mle   [mle]
        ti_mle_height_ratio        exp(-ti_mle_sigma_dist^2)       [mle]

    Parameters
    ----------
    data      : array-like   Expression values across strains for one gene.
    gene_name : str          Name of the gene (used in titles and repr).
    bins      : int          Number of histogram bins (default 40).
    x_max     : float, optional
        Truncation point. Defaults to the observed maximum.

    Example
    -------
    bf = BhuvanFitter(data, gene_name="BRCA1")
    bf.fit("mle")
    print(bf.ti_mle_sigma_dist, bf.ti_mle_height_ratio)
    bf.hist(lines=["mle"])
    """

    _FIT_REGISTRY = {
        "fourparam": "_fit_fourparam",
        "mle":       "_fit_mle",
    }

    def __init__(self, data, gene_name: str, bins: int = 40, x_max=None):
        self.gene_name = gene_name
        self.bins      = bins

        arr = np.asarray(data, dtype=float)
        arr = arr[np.isfinite(arr)]
        if arr.size == 0:
            raise ValueError(f"No finite data values for gene '{gene_name}'.")
        self._data = arr

        self._x_max = float(x_max) if x_max is not None else float(arr.max())

        counts, edges     = np.histogram(arr, bins=bins)
        self.hist_counts  = counts.astype(float)
        self.hist_centers = 0.5 * (edges[:-1] + edges[1:])
        self.hist_edges   = edges

        self.active_fits = {name: False for name in self._FIT_REGISTRY}

        # fourparam result storage
        self.fourparam_y0        = None
        self.fourparam_A         = None
        self.fourparam_x0        = None
        self.fourparam_w         = None
        self.fourparam_sumsquare = None

        # MLE result storage
        self.mle_mu    = None
        self.mle_sigma = None
        self.mle_nll   = None

    # ── Simple statistics ────────────────────────────────────────────────────

    def max(self):    return float(self._data.max())
    def min(self):    return float(self._data.min())
    def mean(self):   return float(self._data.mean())
    def std(self):    return float(self._data.std())
    def median(self): return float(np.median(self._data))

    # ── Truncation index properties — fourparam ──────────────────────────────

    @property
    def ti_fourparam_sigma_dist(self):
        """
        Fourparam σ-distance truncation index.

            (x_max - x0) / (w / sqrt(2))

        Converts the fourparam width parameter w to sigma via w/sqrt(2),
        then returns how many of those sigmas x_max lies above the fitted peak x0.
        Lower = ceiling closer to peak = stronger lethality constraint.
        Biased because the fit is anchored to already-truncated data.
        """
        if not self.active_fits.get("fourparam"):
            raise RuntimeError("fourparam fit has not been run. Call bf.fit('fourparam') first.")
        sigma_fp = self.fourparam_w / np.sqrt(2.0)
        return float((self._x_max - self.fourparam_x0) / sigma_fp)

    @property
    def ti_fourparam_height_ratio(self):
        """
        Fourparam height-ratio truncation index.

            f(x_max) / f(x0)

        Ratio of the fitted histogram curve's value at the ceiling to its value
        at the peak. Bounded [0, 1] when the fit is well-behaved.
        Higher = ceiling nearer to peak = stronger lethality constraint.
        Biased for the same reason as ti_fourparam_sigma_dist.
        """
        if not self.active_fits.get("fourparam"):
            raise RuntimeError("fourparam fit has not been run. Call bf.fit('fourparam') first.")

        x_range = np.linspace(self.hist_edges[0], self.hist_edges[-1], 600)
        peak = self.fourparam_function(x_range).max() - self.fourparam_function(self.min()) #self.fourparam_y0

        tail = self.fourparam_function(self._x_max)       - self.fourparam_function(self.min()) #self.fourparam_y0
        return float(tail / peak)

    # ── Truncation index properties — MLE ────────────────────────────────────

    @property
    def ti_mle_sigma_dist(self):
        """
        MLE σ-distance truncation index.

            (x_max - mu_mle) / sigma_mle

        How many standard deviations of the TRUE underlying distribution
        x_max lies above the true mean. Uses the unbiased MLE estimates
        that account for the missing right tail.
        Lower = ceiling closer to peak = stronger lethality constraint.
        """
        if not self.active_fits.get("mle"):
            raise RuntimeError("MLE fit has not been run. Call bf.fit('mle') first.")
        return float((self._x_max - self.mle_mu) / self.mle_sigma)

    @property
    def ti_mle_height_ratio(self):
        """
        MLE height-ratio truncation index.

            exp(-ti_mle_sigma_dist^2)

        Evaluates the true Gaussian at x_max relative to its peak, using
        the unbiased MLE mu and sigma. Equivalent to f(x_max)/f(mu) for a
        Gaussian. Bounded [0, 1].
        Higher = ceiling nearer to peak = stronger lethality constraint.
        """
        if not self.active_fits.get("mle"):
            raise RuntimeError("MLE fit has not been run. Call bf.fit('mle') first.")
        return float(np.exp(-(self.ti_mle_sigma_dist ** 2)))

    # ── Public dispatch ───────────────────────────────────────────────────────

    def fit(self, fit_name: str):
        """
        Run the specified fit.

        Parameters
        ----------
        fit_name : str   'fourparam' or 'mle'.
        """
        if fit_name not in self._FIT_REGISTRY:
            raise ValueError(
                f"Unknown fit '{fit_name}'. Supported: {list(self._FIT_REGISTRY.keys())}"
            )
        getattr(self, self._FIT_REGISTRY[fit_name])()
        self.active_fits[fit_name] = True

    # ── fourparam fit ─────────────────────────────────────────────────────────

    def _fit_fourparam(self):
        """
        Fit a 4-parameter Gaussian to histogram bin counts via nonlinear
        least squares (TRF method, robust soft-L1 loss).
        """
        x = self.hist_centers
        y = self.hist_counts

        y0_0 = float(y.min())
        A_0  = float(y.max() - y0_0)
        x0_0 = float(x[np.argmax(y)])

        half_max = y0_0 + A_0 / 2.0
        above    = x[y >= half_max]
        if len(above) > 1:
            fwhm = float(above[-1] - above[0])
            w_0  = fwhm / (2.0 * np.sqrt(np.log(2.0)))
        else:
            w_0 = float(np.std(self._data)) * np.sqrt(2.0)
        w_0 = max(w_0, 1e-6)

        p0 = [y0_0, A_0, x0_0, w_0]

        try:
            popt, _ = curve_fit(
                _fourparam_gaussian,
                x, y,
                p0=p0,
                bounds=([-np.inf, 0.0, -np.inf, 1e-6],
                        [ np.inf, np.inf, np.inf, np.inf]),
                method="trf",
                loss="soft_l1",
                max_nfev=10_000,
            )
        except (RuntimeError, ValueError) as exc:
            raise RuntimeError(
                f"fourparam fit failed for gene '{self.gene_name}': {exc}"
            ) from exc

        y0_fit, A_fit, x0_fit, w_fit = popt
        self.fourparam_y0        = float(y0_fit)
        self.fourparam_A         = float(A_fit)
        self.fourparam_x0        = float(x0_fit)
        self.fourparam_w         = float(w_fit)
        residuals                = y - _fourparam_gaussian(x, *popt)
        self.fourparam_sumsquare = float(np.sum(residuals ** 2))

    # ── MLE fit ───────────────────────────────────────────────────────────────

    def _fit_mle(self):
        """
        Fit a right-truncated Gaussian to the raw data via MLE.
        Recovers the true mu and sigma of the underlying distribution,
        accounting for the missing right tail above x_max.
        """
        try:
            mu, sigma, nll = _fit_mle_truncated(self._data, self._x_max)
        except RuntimeError as exc:
            raise RuntimeError(
                f"MLE fit failed for gene '{self.gene_name}': {exc}"
            ) from exc
        self.mle_mu    = mu
        self.mle_sigma = sigma
        self.mle_nll   = nll

    # ── Evaluate fitted curves ────────────────────────────────────────────────

    def fourparam_function(self, x):
        """Evaluate the fitted 4-parameter Gaussian at x."""
        if not self.active_fits.get("fourparam"):
            raise RuntimeError("fourparam fit has not been run.")
        return _fourparam_gaussian(
            np.asarray(x, dtype=float),
            self.fourparam_y0, self.fourparam_A,
            self.fourparam_x0, self.fourparam_w,
        )

    def mle_pdf(self, x):
        """
        Evaluate the MLE-fitted truncated Gaussian PDF at x,
        scaled to histogram counts for overlay purposes.
        """
        if not self.active_fits.get("mle"):
            raise RuntimeError("MLE fit has not been run.")
        x   = np.asarray(x, dtype=float)
        mu, sigma = self.mle_mu, self.mle_sigma
        z_max = (self._x_max - mu) / sigma
        pdf   = norm.pdf(x, loc=mu, scale=sigma) / (ndtr(z_max) + 1e-300)
        bin_width   = np.diff(self.hist_edges).mean()
        total_count = self.hist_counts.sum()
        return pdf * total_count * bin_width

    def mle_true_pdf(self, x):
        """
        Evaluate the TRUE (untruncated) Gaussian PDF at x, scaled to
        histogram counts. Shows what the distribution would look like
        without the lethality ceiling.
        """
        if not self.active_fits.get("mle"):
            raise RuntimeError("MLE fit has not been run.")
        x = np.asarray(x, dtype=float)
        bin_width   = np.diff(self.hist_edges).mean()
        total_count = self.hist_counts.sum()
        return norm.pdf(x, loc=self.mle_mu, scale=self.mle_sigma) * total_count * bin_width

    # ── Visualisation ─────────────────────────────────────────────────────────

    def hist(self, lines=None, show_true_gaussian=False):
        """
        Plot the histogram and optionally overlay fitted curves.

        Parameters
        ----------
        lines : list of str, optional
            Fits to overlay: ['fourparam'], ['mle'], or both.
        show_true_gaussian : bool
            If True and MLE is active, also draws the untruncated true
            Gaussian as a dashed line.
        """
        fig, ax = plt.subplots(figsize=(9, 5))

        ax.bar(
            self.hist_centers,
            self.hist_counts,
            width=np.diff(self.hist_edges).mean(),
            color="steelblue", alpha=0.6,
            label="Observed", zorder=2,
        )

        x_smooth = np.linspace(self.hist_edges[0], self.hist_edges[-1], 600)

        if lines:
            for fit_name in lines:
                if fit_name not in self._FIT_REGISTRY:
                    print(f"Warning: '{fit_name}' not recognised — skipping.")
                    continue
                if not self.active_fits.get(fit_name):
                    print(f"Warning: '{fit_name}' not fitted yet — skipping.")
                    continue

                if fit_name == "fourparam":
                    y_curve = self.fourparam_function(x_smooth)
                    label   = (
                        f"4-param Gaussian (histogram fit)\n"
                        f"A={self.fourparam_A:.3g}, x0={self.fourparam_x0:.3g}, "
                        f"w={self.fourparam_w:.3g}, y0={self.fourparam_y0:.3g}\n"
                        f"ti_fourparam_sigma_dist={self.ti_fourparam_sigma_dist:.4f}σ  "
                        f"ti_fourparam_height_ratio={self.ti_fourparam_height_ratio:.4f}"
                    )
                    ax.plot(x_smooth, y_curve, color="crimson",
                            linewidth=2, label=label, zorder=3)

                    # Horizontal lines at the two y-values used in ti_fourparam_height_ratio
                    y_peak = self.fourparam_function(self.fourparam_x0) - self.fourparam_y0
                    y_tail = self.fourparam_function(self._x_max)       - self.fourparam_y0


                    ax.axhline(self.fourparam_function(self.min()), color="crimson", linestyle="--", linewidth=1, alpha=0.6, label=f"baseline [f(min)] = {self.fourparam_function(self.min()):.3g}", zorder=3)
                    ax.axhline(self.fourparam_function(self.fourparam_x0), color="crimson", linestyle=":", linewidth=1, alpha=0.6, label=f"offset [f(x0)] = {self.fourparam_y0:.3g}", zorder=3)


                elif fit_name == "mle":
                    y_curve = self.mle_pdf(x_smooth)
                    label   = (
                        f"MLE truncated Gaussian\n"
                        f"μ={self.mle_mu:.3g}, σ={self.mle_sigma:.3g}\n"
                        f"ti_mle_sigma_dist={self.ti_mle_sigma_dist:.4f}σ  "
                        f"ti_mle_height_ratio={self.ti_mle_height_ratio:.4f}"
                    )
                    ax.plot(x_smooth, y_curve, color="darkorange",
                            linewidth=2, label=label, zorder=3)

        if show_true_gaussian and self.active_fits.get("mle"):
            x_ext = np.linspace(self.hist_edges[0], self.mle_mu + 4 * self.mle_sigma, 800)
            ax.plot(x_ext, self.mle_true_pdf(x_ext), color="seagreen",
                    linewidth=2, linestyle="--",
                    label="True Gaussian (untruncated)", zorder=3)

        if self.active_fits.get("mle"):
            ax.axvline(self._x_max, color="black", linestyle=":",
                       linewidth=1.5, label=f"x_max = {self._x_max:.3g}", zorder=4)
            ax.axvline(self.mle_mu, color="darkorange", linestyle=":",
                       linewidth=1.5, alpha=0.6,
                       label=f"μ_MLE = {self.mle_mu:.3g}", zorder=4)

        ax.set_title(self.gene_name, fontsize=14, fontweight="bold")
        ax.set_xlabel("Gene Expression Level", fontsize=12)
        ax.set_ylabel("Frequency (bin count) ", fontsize=12)
        ax.legend(fontsize=9)
        ax.grid(True, linestyle="--", alpha=0.4, zorder=1)
        plt.tight_layout()
        plt.show()
        return ax

    # ── Dunder ────────────────────────────────────────────────────────────────

    def __repr__(self):
        active = [k for k, v in self.active_fits.items() if v]
        parts  = [f"BhuvanFitter(gene='{self.gene_name}', n={len(self._data)}, active_fits={active}"]
        if self.active_fits.get("fourparam"):
            parts.append(
                f"  ti_fourparam_sigma_dist={self.ti_fourparam_sigma_dist:.4f}σ"
                f", ti_fourparam_height_ratio={self.ti_fourparam_height_ratio:.4f}"
            )
        if self.active_fits.get("mle"):
            parts.append(
                f"  ti_mle_sigma_dist={self.ti_mle_sigma_dist:.4f}σ"
                f", ti_mle_height_ratio={self.ti_mle_height_ratio:.4f}"
            )
        parts.append(")")
        return "\n".join(parts)


## 5. Detecting Genes with a -1 Spike

Expression values of exactly **-1** are a sentinel in this dataset — they represent
strains where the gene was not expressed (below detection threshold). Genes with a
large spike at -1 have a mixed distribution and should be treated with caution or
excluded before computing truncation indices.

`has_minus_one_peak` flags any gene where ≥ `min_fraction` of observations fall
at or below `−1 + tolerance`.

In [ ]:
def has_minus_one_peak(data_vector, tolerance=0.25, min_fraction=0.20):
    """
    Returns True if the data array contains a significant spike at -1.

    Parameters
    ----------
    data_vector  : array-like   1D array of gene expression values.
    tolerance    : float        Half-width of the window around -1 (default 0.25).
    min_fraction : float        Minimum proportion of values in that window to
                                trigger the flag (default 0.20 = 20%).

    Returns
    -------
    bool   True if a significant -1 spike is detected.
    """
    clean_data = np.asarray(data_vector, dtype=float)
    clean_data = clean_data[np.isfinite(clean_data)]

    if clean_data.size == 0:
        return False

    upper_bound         = -1.0 + tolerance
    points_at_minus_one = np.sum(clean_data <= upper_bound)
    fraction            = points_at_minus_one / clean_data.size

    return fraction >= min_fraction


## 6. Batch Analysis Functions

These two functions are **independent** — each runs only its own model and
outputs only its own metrics. They can be joined on `gene` if you want to
compare MLE vs fourparam estimates side by side.

### `compute_mle_table`
Runs the **MLE** fit only. Outputs:
- `ti_mle_sigma_dist` — `(x_max − μ_mle) / σ_mle`
- `ti_mle_height_ratio` — `exp(−ti_mle_sigma_dist²)`
- Plus MLE model parameters: `mu_mle`, `sigma_mle`, `mle_nll`

### `compute_fourparam_table`
Runs the **fourparam** fit only. Outputs:
- `ti_fourparam_sigma_dist` — `(x_max − x0) / (w / √2)`
- `ti_fourparam_height_ratio` — `f(x_max) / f(x0)`
- Plus fourparam model parameters: `y0`, `A`, `x0`, `w`, `fourparam_sumsquare`
- Plus `has_minus_one_peak` flag

In [ ]:
def compute_mle_table(df, bins=40, x_max_percentile=100):
    """
    Fit a right-truncated Gaussian via MLE for every gene in the dataframe.

    Each gene is fitted independently. Only the MLE model is run.

    Parameters
    ----------
    df : pd.DataFrame
        Rows = strains, columns = genes. Values are expression levels.
    bins : int
        Histogram bins passed to BhuvanFitter (default 40).
    x_max_percentile : float
        Percentile of the data to use as the truncation point (default 100 =
        true maximum). Use 99 for robustness against single extreme outliers.

    Returns
    -------
    pd.DataFrame
        One row per gene, sorted by ti_mle_sigma_dist ascending
        (most truncated genes first). Columns:
            gene, mu_mle, sigma_mle, x_max,
            ti_mle_sigma_dist, ti_mle_height_ratio,
            mle_nll, n_obs, fit_success
    """
    records = []

    for gene in df.columns:
        data = df[gene].astype(float).values
        data = data[np.isfinite(data)]

        if len(data) < 10:
            records.append({
                "gene": gene, "mu_mle": np.nan, "sigma_mle": np.nan,
                "x_max": np.nan,
                "ti_mle_sigma_dist": np.nan,
                "ti_mle_height_ratio": np.nan,
                "mle_nll": np.nan, "n_obs": len(data), "fit_success": False
            })
            continue

        x_max = float(np.percentile(data, x_max_percentile))

        try:
            bf = BhuvanFitter(data, gene_name=gene, bins=bins, x_max=x_max)
            bf.fit("mle")
            records.append({
                "gene":               gene,
                "mu_mle":             bf.mle_mu,
                "sigma_mle":          bf.mle_sigma,
                "x_max":              x_max,
                "ti_mle_sigma_dist":  bf.ti_mle_sigma_dist,
                "ti_mle_height_ratio":bf.ti_mle_height_ratio,
                "mle_nll":            bf.mle_nll,
                "n_obs":              len(data),
                "fit_success":        True,
            })
        except RuntimeError as e:
            warnings.warn(f"Gene '{gene}' failed: {e}")
            records.append({
                "gene": gene, "mu_mle": np.nan, "sigma_mle": np.nan,
                "x_max": x_max,
                "ti_mle_sigma_dist": np.nan,
                "ti_mle_height_ratio": np.nan,
                "mle_nll": np.nan, "n_obs": len(data), "fit_success": False
            })

    result = pd.DataFrame(records)
    result = result.sort_values("ti_mle_sigma_dist", ascending=True).reset_index(drop=True)
    return result


def compute_fourparam_table(df, bins=40, x_max_percentile=100):
    """
    Fit a 4-parameter Gaussian to histogram counts for every gene in the dataframe.

    Each gene is fitted independently. Only the fourparam model is run.

    Parameters
    ----------
    df : pd.DataFrame
        Rows = strains, columns = genes. Values are expression levels.
    bins : int
        Histogram bins for the fourparam fit (default 40).
    x_max_percentile : float
        Percentile of the data to use as the truncation point (default 100).
        Use 99 for robustness against single extreme outliers.

    Returns
    -------
    pd.DataFrame
        One row per gene in insertion order. Columns:
            gene, has_minus_one_peak, y0, A, x0, w, fourparam_sumsquare,
            ti_fourparam_sigma_dist, ti_fourparam_height_ratio,
            n_obs, fit_success
    """
    records = []

    for gene in df.columns:
        data = df[gene].astype(float).values
        data = data[np.isfinite(data)]

        if len(data) < 10:
            records.append({
                "gene": gene,
                "has_minus_one_peak": has_minus_one_peak(df[gene]),
                "y0": np.nan, "A": np.nan, "x0": np.nan, "w": np.nan,
                "fourparam_sumsquare": np.nan,
                "ti_fourparam_sigma_dist": np.nan,
                "ti_fourparam_height_ratio": np.nan,
                "n_obs": len(data), "fit_success": False
            })
            continue

        x_max = float(np.percentile(data, x_max_percentile))

        try:
            bf = BhuvanFitter(data, gene_name=gene, bins=bins, x_max=x_max)
            bf.fit("fourparam")
            records.append({
                "gene":                     gene,
                "has_minus_one_peak":       has_minus_one_peak(df[gene]),
                "y0":                       bf.fourparam_y0,
                "A":                        bf.fourparam_A,
                "x0":                       bf.fourparam_x0,
                "w":                        bf.fourparam_w,
                "fourparam_sumsquare":      bf.fourparam_sumsquare,
                "ti_fourparam_sigma_dist":  bf.ti_fourparam_sigma_dist,
                "ti_fourparam_height_ratio":bf.ti_fourparam_height_ratio,
                "n_obs":                    len(data),
                "fit_success":              True,
            })
        except RuntimeError as e:
            warnings.warn(f"Fourparam fit for gene '{gene}' failed: {e}")
            records.append({
                "gene": gene,
                "has_minus_one_peak": has_minus_one_peak(df[gene]),
                "y0": np.nan, "A": np.nan, "x0": np.nan, "w": np.nan,
                "fourparam_sumsquare": np.nan,
                "ti_fourparam_sigma_dist": np.nan,
                "ti_fourparam_height_ratio": np.nan,
                "n_obs": len(data), "fit_success": False
            })

    return pd.DataFrame(records)


## 7. Visualising the Truncation Index Distribution

After running `compute_mle_table`, the plots below give a landscape view:

- **Left**: histogram of `ti_mle_sigma_dist` across all genes — useful for spotting bimodality.
- **Right**: ranked bar chart of the most-truncated genes — your primary hit list.

In [ ]:
def plot_truncation_index_distribution(mle_table, top_n=20):
    """
    Plot the distribution of MLE sigma-distance TI values across all genes
    and highlight the top_n most truncated genes.

    Parameters
    ----------
    mle_table : pd.DataFrame   Output of compute_mle_table.
    top_n     : int            Number of most-truncated genes to label.
    """
    valid = mle_table.dropna(subset=["ti_mle_sigma_dist"])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(valid["ti_mle_sigma_dist"], bins=40,
                 color="steelblue", alpha=0.7, edgecolor="white")
    axes[0].set_xlabel("ti_mle_sigma_dist (σ units)", fontsize=12)
    axes[0].set_ylabel("Number of genes", fontsize=12)
    axes[0].set_title("Distribution of MLE Truncation Indices", fontsize=13, fontweight="bold")
    axes[0].grid(True, linestyle="--", alpha=0.4)

    top    = valid.head(top_n)
    colors = plt.cm.RdYlGn(np.linspace(0.1, 0.6, len(top)))
    axes[1].barh(top["gene"], top["ti_mle_sigma_dist"], color=colors)
    axes[1].set_xlabel("ti_mle_sigma_dist (σ units)", fontsize=12)
    axes[1].set_title(f"Top {top_n} Most Truncated Genes (MLE)", fontsize=13, fontweight="bold")
    axes[1].invert_yaxis()
    axes[1].grid(True, linestyle="--", alpha=0.4, axis="x")

    plt.tight_layout()
    plt.show()


## 8. Loading the Data

The raw data lives in a CSV on Google Drive. Each **row** is a strain and each
**column** is a gene. After transposing, rows = strains and columns = genes,
which is the expected input format for all BhuvanFitter functions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

try:
    df = pd.read_csv('/content/drive/MyDrive/Supplementary Data 1_csv.csv')
    print('CSV loaded successfully!')
    df = df.set_index('strain')
    df = df.T   # rows = strains, columns = genes
    print(f'Shape after transpose: {df.shape}')
except FileNotFoundError:
    print('Error: CSV file not found. Check the path.')
except Exception as e:
    print(f'An error occurred: {e}')


## 9. Single-Gene Example

Before running the full batch, inspect a single gene to verify both fits look
sensible. The legend on the histogram reports all four TI metrics for direct
side-by-side comparison.

In [ ]:
gene = 'w24401_C27F2.9.2'

print(f"has_minus_one_peak: {has_minus_one_peak(df[gene])}")

expressed = df[gene].astype(float).values

bf = BhuvanFitter(expressed, gene_name=gene)
bf.fit("fourparam")
bf.fit("mle")

bf.hist(lines=['fourparam', 'mle'], show_true_gaussian=True)
print(bf)

print(f"\nFourparam metrics:")
print(f"  ti_fourparam_sigma_dist:   {bf.ti_fourparam_sigma_dist:.4f} σ")
print(f"  ti_fourparam_height_ratio: {bf.ti_fourparam_height_ratio:.4f}")
print(f"\nMLE metrics:")
print(f"  ti_mle_sigma_dist:         {bf.ti_mle_sigma_dist:.4f} σ")
print(f"  ti_mle_height_ratio:       {bf.ti_mle_height_ratio:.4f}")


## 10. Batch Run — All Genes

Each function runs independently. `compute_mle_table` runs only the MLE fit;
`compute_fourparam_table` runs only the fourparam fit. Each table contains only
its own model's parameters and TI metrics. They share only the `gene` column,
and can be merged on that if you want a unified comparison table.

In [ ]:
# print("Computing MLE truncation index table for all genes...")
# mle_table = compute_mle_table(df)
# print(f"Done. Shape: {mle_table.shape}")
# display(mle_table)

# print("Computing fourparam metrics table for all genes...")
# fourparam_table = compute_fourparam_table(df)
# print(f"Done. Shape: {fourparam_table.shape}")
# display(fourparam_table)


## 11. Optional: Merge MLE and Fourparam Tables

If you want to compare MLE vs fourparam estimates side by side for each gene,
merge on `gene`. The four TI columns will sit next to each other for easy comparison.

In [ ]:
# # Optional — merge the two tables on gene for side-by-side comparison
# combined_ti = mle_table.merge(
#     fourparam_table[['gene', 'ti_fourparam_sigma_dist', 'ti_fourparam_height_ratio']],
#     on='gene',
#     how='left'
# )
# display(combined_ti[['gene', 'ti_mle_sigma_dist', 'ti_fourparam_sigma_dist',
#                       'ti_mle_height_ratio', 'ti_fourparam_height_ratio']])


## 12. Visualise the Truncation Index Distribution

In [ ]:
# plot_truncation_index_distribution(mle_table, top_n=20)


## 13. Combining Expression Data with Fourparam Fit Metrics

Append the per-gene fourparam metrics as new rows at the bottom of the expression
DataFrame for export or downstream analysis.

In [ ]:
# fourparam_metrics_transposed_df = fourparam_table.set_index('gene').T
# combined_df = pd.concat([df, fourparam_metrics_transposed_df], axis=0)

# print("Shape of the combined DataFrame:", combined_df.shape)
# display(combined_df)


## 14. Gene Sets of Interest

Curated gene sets for downstream comparison against the truncation index rankings:

- **`mco_dev`** / **`mco_behavior`**: multi-copy overexpression phenotype genes
- **`lof_dev`** / **`lof_behavior`**: loss-of-function phenotype genes

In [ ]:
mco_dev = [
    "chaf-2", "cle-1", "dip-2", "eva-1", "pat-3", "mrps-6",
    "ncam-1", "F43G9.12", "rcan-1", "rrp-1", "nrd-1", "Y105E8A.1",
    "hlh-34", "sod-1", "D1037.1", "unc-26", "Y74C10AL.2", "trpp-10",
    "K02C4.3", "pfk-1.1"
]

mco_behavior = [
    "chaf-2", "cle-1", "eva-1", "ltn1", "ncam-1", "F43G9.12",
    "rrp-1", "hlh-34", "D1037.1"
]

lof_dev = [
    "atp-3", "cbs-1", "cct-8", "cle-1", "dip-2", "dnsn-1",
    "stc-1", "mrpl-39", "mrps-6", "ncam-1", "rcan-1", "rrp-1",
    "nrd-1", "sod-1"
]

lof_behavior = [
    "B0024.15", "cle-1", "eva-1", "mtq-2", "pes-4", "pdxk-1",
    "pad-2", "rnt-1", "unc-26", "ubc-14", "wdr-4"
]


## 15. Mapping Short Gene Names to DataFrame Identifiers

The supplementary Excel file maps short gene names (e.g. `"chaf-2"`) to the
full strain-derived identifiers used in the expression DataFrame
(e.g. `"w13643_chaf-2.1"`).

In [ ]:
pierce = pd.read_excel('Supplementary Data 1 trunc 20250702.xlsx')
pierce = pierce[['gene', 'GeneName']]

genenames = pierce['gene'].str[4:-5]
genenames = genenames.str.replace('_', '.')
genenames = genenames.str.replace('.', '_', n=1)

pierce['converted_gene'] = genenames

# Build dictionary with LISTS of matches
gene_dict = (
    pierce.groupby('GeneName')['converted_gene']
    .apply(list)
    .to_dict()
)

converted_mco_dev = {
    g: gene_dict[g]
    for g in mco_dev
    if g in gene_dict
}

converted_mco_behavior = {
    g: gene_dict[g]
    for g in mco_behavior
    if g in gene_dict
}

converted_lof_dev = {
    g: gene_dict[g]
    for g in lof_dev
    if g in gene_dict
}

converted_lof_behavior = {
    g: gene_dict[g]
    for g in lof_behavior
    if g in gene_dict
}

print(f"mco_dev      → {len(converted_mco_dev)}/{len(mco_dev)} genes mapped")
print(f"mco_behavior → {len(converted_mco_behavior)}/{len(mco_behavior)} genes mapped")
print(f"lof_dev      → {len(converted_lof_dev)}/{len(lof_dev)} genes mapped")
print(f"lof_behavior → {len(converted_lof_behavior)}/{len(lof_behavior)} genes mapped")

# import json

# # # Export fourparam_table as a parquet file
# # fourparam_table.to_parquet('/content/drive/MyDrive/fourparam_table.parquet')
# # print('fourparam_table exported to fourparam_table.parquet')

# # # Export mle_table as a parquet file
# # mle_table.to_parquet('/content/drive/MyDrive/mle_table.parquet')
# # print('mle_table exported to mle_table.parquet')

# # Combine converted gene lists into a single dictionary
# genes_of_interest_dict = {
#     'mco_dev': converted_mco_dev,
#     'mco_behavior': converted_mco_behavior,
#     'lof_dev': converted_lof_dev,
#     'lof_behavior': converted_lof_behavior
# }

# # Export the combined genes of interest dictionary as a JSON file
# with open('/content/drive/MyDrive/genes_of_interest.json', 'w') as f:
#     json.dump(genes_of_interest_dict, f, indent=4)
# print('Combined genes of interest dictionary exported to genes_of_interest.json')

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import json

# Define the base path to your Google Drive
drive_path = '/content/drive/MyDrive/bhuvan research project/' # Adjust this path if your files are in a specific folder

# Load fourparam_table.parquet
try:
    fourparam_df = pd.read_parquet(f'{drive_path}fourparam_table.parquet')
    print('fourparam_table.parquet loaded successfully.')
    display(fourparam_df.head())
except FileNotFoundError:
    print(f'Error: fourparam_table.parquet not found at {drive_path}fourparam_table.parquet. Please check the path.')

# Load mle_table.parquet
try:
    mle_df = pd.read_parquet(f'{drive_path}mle_table.parquet')
    print('mle_table.parquet loaded successfully.')
    display(mle_df.head())
except FileNotFoundError:
    print(f'Error: mle_table.parquet not found at {drive_path}mle_table.parquet. Please check the path.')

# Load genes_of_interest.json
try:
    with open(f'{drive_path}genes_of_interest.json', 'r') as f:
        genes_of_interest = json.load(f)
    print('genes_of_interest.json loaded successfully.')
    print(f'Number of genes of interest: {len(genes_of_interest)}')
    print(f'First 5 genes: {list(genes_of_interest.keys())[:5]}')
except FileNotFoundError:
    print(f'Error: genes_of_interest.json not found at {drive_path}genes_of_interest.json. Please check the path.')
except json.JSONDecodeError:
    print(f'Error: Could not decode genes_of_interest.json. Ensure it is a valid JSON file.')

def analyze_best_isoforms(gene_isoform_map):
    """
    For each gene in gene_isoform_map, fits a fourparam Gaussian to each isoform,
    selects the best isoform by highest height ratio, and summarizes results.

    Parameters
    ----------
    gene_isoform_map : dict
        Keys are gene names, values are lists of isoform identifiers.
        e.g. {'chaf-2': ['w13643_chaf-2.1', 'w13643_chaf-2.2'], ...}

    Returns
    -------
    dict
        best_genes_to_use: one entry per gene with keys 'gene', 'ratio', 'sd'.
    """
    best_genes_to_use = {}

    for gene in gene_isoform_map:

        maxratio = -100
        cur_max_name = ""
        best_sd = None

        for isoform in gene_isoform_map[gene]:

            print(has_minus_one_peak(df[isoform]))

            idk = df[isoform][df[isoform] > -0.75].astype(float).values

            genefitter = BhuvanFitter(
                idk,
                gene_name=isoform + ": isoform of " + gene
            )

            genefitter.fit("fourparam")
            genefitter.hist(lines=['fourparam'])

            cur_ratio = genefitter.ti_fourparam_height_ratio
            cur_sd    = genefitter.ti_fourparam_sigma_dist

            if cur_ratio > maxratio:
                maxratio     = cur_ratio
                best_sd      = cur_sd
                cur_max_name = isoform

        print("Probably best isoform to use: " + cur_max_name)

        best_genes_to_use[gene] = {
            "gene":  cur_max_name,
            "ratio": maxratio,
            "sd":    best_sd,
        }

        print("=" * 93)

    # ── Summary ───────────────────────────────────────────────────────────────
    ratios = [info["ratio"] for info in best_genes_to_use.values()]
    sds    = [info["sd"]    for info in best_genes_to_use.values()]

    for gene, info in best_genes_to_use.items():
        print(f"{gene}: ratio={info['ratio']:.4f}, sd={info['sd']:.4f}")

    n = len(best_genes_to_use)
    print(f"\nAverage height ratio: {np.mean(ratios):.4f}  (std: {np.std(ratios, ddof=1):.4f})")
    print(f"Average sd:           {np.mean(sds):.4f}  (std: {np.std(sds,    ddof=1):.4f})")

    return best_genes_to_use

#analyze_best_isoforms(converted_mco_behavior)

#analyze_best_isoforms(converted_mco_behavior)

#analyze_best_isoforms(converted_lof_dev)

#analyze_best_isoforms(converted_lof_behavior)

trial = BhuvanFitter(df['w22333_Y17G9B.5.1'], gene_name='w16468_C16A11.6.2', bins=40)
trial.fit("fourparam")
trial.hist(lines=['fourparam'])

print(trial.fourparam_y0)

print(f"\nFourparam fit parameters for gene '{trial.gene_name}':")
print(f"  x_max: {trial._x_max}")
print(f"  y0: {trial.fourparam_y0}")
print(f"  A: {trial.fourparam_A}")
print(f"  x0: {trial.fourparam_x0}")
print(f"  w: {trial.fourparam_w}")

# Calculate f(x0) and f(x_max) using the _fourparam_gaussian function
peak_value = _fourparam_gaussian(trial.fourparam_x0, trial.fourparam_y0, trial.fourparam_A, trial.fourparam_x0, trial.fourparam_w)
tail_value = _fourparam_gaussian(trial._x_max, trial.fourparam_y0, trial.fourparam_A, trial.fourparam_x0, trial.fourparam_w)

print(f"\nCalculated f(x0) (value at peak): {peak_value}")
print(f"Calculated f(x_max) (value at ceiling): {tail_value}")
print(f"Calculated height ratio (f(x_max) / f(x0)): {tail_value / peak_value}")
